# EMSN194 Max Water Depth — COG and Tiles

Converts the Copernicus EMS Max Water Depth GeoTIFF into:
- **COG** (Cloud Optimized GeoTIFF) in Web Mercator
- **Visual tiles** — colorized XYZ tiles for map display
- **Value tiles** — Terrain RGB encoded for client-side hover lookup

**Input:** `data/EMSN194_STD_AOI01_P06TMFL_MaxWaterDepth_v01.tif` (UTM 22S, 30m)  
**Outputs:** `output/` — COG, tiles, metadata.json

**Requires:** GDAL (`gdalwarp`, `gdal_translate`, `gdaldem`, `gdal2tiles.py`)

In [1]:
import subprocess
import json
from pathlib import Path

# Paths (run from release/v1/2024/)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
IN_TIF = DATA_DIR / "EMSN194_STD_AOI01_P06TMFL_MaxWaterDepth_v01.tif"
OUT_3857 = OUTPUT_DIR / "maxwaterdepth_3857.tif"
OUT_COG = OUTPUT_DIR / "maxwaterdepth_cog.tif"
OUT_COLORIZED = OUTPUT_DIR / "maxwaterdepth_colorized.tif"
OUT_VALUE_ENCODED = OUTPUT_DIR / "maxwaterdepth_value_encoded.tif"
TILES_VISUAL = OUTPUT_DIR / "tiles_visual"
TILES_VALUES = OUTPUT_DIR / "tiles_values"
COLORS_VISUAL = DATA_DIR / "maxwaterdepth_visual_colors.txt"
COLORS_VALUE = DATA_DIR / "maxwaterdepth_value_encoding_colors.txt"
METADATA_JSON = OUTPUT_DIR / "maxwaterdepth_metadata.json"

# Terrain RGB encoding params (for frontend decode: value = (R*256² + G*256 + B) * scale + offset)
MAXWATERDEPTH_SCALE = 0.01
MAXWATERDEPTH_OFFSET = 0.0
MAXWATERDEPTH_UNIT = "m"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not IN_TIF.exists():
    raise FileNotFoundError(f"Expected input: {IN_TIF}")

## 1. Reproject to Web Mercator (EPSG:3857)

Source is UTM zone 22S (EPSG:32722). Tiles require Web Mercator.

In [3]:
subprocess.run([
    "gdalwarp", "-t_srs", "EPSG:3857", "-r", "near",
    str(IN_TIF), str(OUT_3857)
], check=True)
print("Reprojected to Web Mercator:", OUT_3857)

Creating output file that is 933P x 2301L.
Using internal nodata values (e.g. -3.40282e+38) for image data/EMSN194_STD_AOI01_P06TMFL_MaxWaterDepth_v01.tif.
Copying nodata values from source data/EMSN194_STD_AOI01_P06TMFL_MaxWaterDepth_v01.tif to destination output/maxwaterdepth_3857.tif.
Processing EMSN194_STD_AOI01_P06TMFL_MaxWaterDepth_v01.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
Reprojected to Web Mercator: output/maxwaterdepth_3857.tif


## 2. Convert to COG

Use `RESAMPLING=AVERAGE` for numeric depth data.

In [4]:
subprocess.run([
    "gdal_translate", str(OUT_3857), str(OUT_COG),
    "-of", "COG",
    "-co", "COMPRESS=DEFLATE",
    "-co", "OVERVIEWS=AUTO",
    "-co", "RESAMPLING=AVERAGE"
], check=True)
print("Created COG:", OUT_COG)

Input file size is 933, 2301
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG: output/maxwaterdepth_cog.tif


## 3. Colorize for visual tiles

Map depth values (0–10 m) to a gradient for map display.

In [5]:
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VISUAL), str(OUT_COLORIZED)
], check=True)
print("Colorized for display:", OUT_COLORIZED)

0...10...20...30...40...50...60...70...80...90...100 - done.
Colorized for display: output/maxwaterdepth_colorized.tif


## 4. Generate visual XYZ tiles

In [ ]:
TILES_VISUAL.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_COLORIZED), str(TILES_VISUAL)
], check=True)
print("Visual tiles written to:", TILES_VISUAL)

## 5. Value tiles (Terrain RGB encoding)

Encode depth values in RGB so the frontend can decode: `value = (R*256² + G*256 + B) * scale + offset`

In [ ]:
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VALUE), str(OUT_VALUE_ENCODED)
], check=True)

TILES_VALUES.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_VALUE_ENCODED), str(TILES_VALUES)
], check=True)
print("Value tiles written to:", TILES_VALUES)

## 6. Write metadata for frontend

Store scale, offset, and unit so the frontend can decode value tiles.

In [ ]:
metadata = {
    "layer": "maxwaterdepth",
    "description": "Max water depth (m) — EMSN194 flood delineation, Porto Alegre May 2024",
    "encoding": {
        "type": "terrain_rgb",
        "scale": MAXWATERDEPTH_SCALE,
        "offset": MAXWATERDEPTH_OFFSET,
        "unit": MAXWATERDEPTH_UNIT,
        "decode_formula": "value = (R * 256 * 256 + G * 256 + B) * scale + offset"
    }
}
with open(METADATA_JSON, "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata written to:", METADATA_JSON)